# Tidal modulation of sewer seepage

The demonstration problem in one figure: the coastal tide forces the water table,
the water table controls whether groundwater stands above the pipe invert, and the
seepage into the sewer follows.

Three measures are plotted on stacked panels sharing one time axis rather than
overlaid on twin vertical scales. They carry unrelated units, so a dual axis would
invite reading amplitude and crossing points that are not there.

The finest coupling interval available is used, because it samples the semidiurnal
signal most densely &mdash; at 15-minute coupling the 89-day window holds 8,544
exchanges.

In [ ]:
get_ipython().run_line_magic('matplotlib', 'inline')
import sys
import pathlib as pl

import numpy as np
import matplotlib.pyplot as plt

import flopy.plot.styles as styles

In [ ]:
sys.path.append("../common")
from liss_settings import (
    get_scenario_name,
    get_results_path,
    fig_ext,
    transparent,
)

#### Scenario and windows

In [ ]:
# ---- scenario -----------------------------------------------------------
domain = "gp"
resolution = "high"               # finest grid resolves the tidal signal best
mf_couple_freq_hours = 0.25       # 15-minute coupling: 8,544 exchanges over 89 d
n_connections = 244
# --------------------------------------------------------------------------

# A 15-day overview spans a full spring-neap cycle (~14.8 d); the detail window is
# a quiescent stretch where the semidiurnal signal is the only thing happening and
# can actually be seen. In the overview it is a ripple on a much larger drift.
WIN_START_D, WIN_LEN_D = 30.0, 15.0
ZOOM_START_D, ZOOM_LEN_D = 32.0, 3.0

# gp_sewer.inp geometry, for the unit sewer standards use
PIPE_MILES = 56638.0 / 5280.0     # 56,638 ft of pipe
PIPE_D_IN = 8.0                   # median diameter
GAL2FT3 = 0.133681

scenario = get_scenario_name(domain, resolution, mf_couple_freq_hours, n_connections)
ws = get_results_path(domain, resolution, mf_couple_freq_hours, n_connections)
dt_h = mf_couple_freq_hours

fig_ws = pl.Path("figures")
fig_ws.mkdir(exist_ok=True, parents=True)

print("scenario:", scenario)
print("results :", ws)
assert ws.is_dir(), f"{ws} does not exist - run step2 for this scenario first"

#### Load the exchange arrays

`swmm_q` holds the MODFLOW well flux, which is stored as $-Q$; positive $Q$ is
into the sewer, so the sign is flipped here. `chd_elev` is the constant-head
boundary elevation mapped from the D-Flow FM water level, and its domain mean is
used as the tidal signal. `pipe_state` columns are
`[inactive cells, disconnected junctions, dry pipes]`.

In [ ]:
def stack(path):
    """Load a per-step .npz written by step2 as a (nstep, n) array."""
    z = np.load(path)
    return np.stack([z[k] for k in sorted(z.files, key=int)])


chd = stack(ws / "chd_elev.npz")
q = stack(ws / "swmm_q.npz")
state = stack(ws / "pipe_state.npz")

nstep = chd.shape[0]
t_d = np.arange(1, nstep + 1) * dt_h / 24.0

tide = chd.mean(axis=1)
seep_ft3d = -q.sum(axis=1)
seep_rate = seep_ft3d / GAL2FT3 / (PIPE_D_IN * PIPE_MILES)
connected = (n_connections - state[:, 1]) / n_connections * 100.0

print(f"{nstep:,} coupling steps, {t_d[-1]:.1f} d")
print(f"  tide      {tide.min():+.2f} to {tide.max():+.2f} ft")
print(f"  seepage   {seep_rate.min():+.1f} to {seep_rate.max():+.1f} gpd/in-dia/mile")
print(f"  connected {connected.min():.1f} to {connected.max():.1f} %")

#### Separating the tidal band from the sub-tidal drift

The raw correlation between tide and seepage is near zero, which does **not** mean
the tide has no effect: the seepage series is dominated by slower variation
(spring-neap and surge) that the semidiurnal signal rides on. A crude band-pass
&mdash; the difference of two running means &mdash; isolates the tidal band.

The lag search is restricted to one tidal period. The cross-correlation of two
near-periodic signals has near-equal maxima at every multiple of the period, so an
unrestricted `argmax` returns an essentially arbitrary multiple of 12.42 h.

In [ ]:
M2_H = 12.4206     # principal lunar semidiurnal period


def band(x, lo_h, hi_h):
    """Difference of running means, keeping periods between lo_h and hi_h."""
    def run(n_h):
        n = max(1, int(round(n_h / dt_h)))
        return np.convolve(x, np.ones(n) / n, mode="same")
    return run(lo_h) - run(hi_h)


def lag_within_one_period(x, y):
    """Lag of peak cross-correlation, searched over +/- half a tidal period."""
    a, b = x - x.mean(), y - y.mean()
    xc = np.correlate(b, a, mode="full")
    lags = np.arange(-len(a) + 1, len(a)) * dt_h
    sel = np.abs(lags) <= M2_H / 2.0
    return lags[sel][np.argmax(xc[sel])]


m = (t_d >= WIN_START_D) & (t_d < WIN_START_D + WIN_LEN_D)
z = (t_d >= ZOOM_START_D) & (t_d < ZOOM_START_D + ZOOM_LEN_D)

t_b = band(tide, 3.0, 30.0)[m]
s_b = band(seep_rate, 3.0, 30.0)[m]

print(f"window {WIN_START_D:.0f}-{WIN_START_D + WIN_LEN_D:.0f} d")
print(f"  raw correlation            {np.corrcoef(tide[m], seep_rate[m])[0, 1]:+.3f}"
      "   (dominated by sub-tidal drift)")
print(f"  semidiurnal band           {np.corrcoef(t_b, s_b)[0, 1]:+.3f}, "
      f"lag {lag_within_one_period(t_b, s_b):+.2f} h")
print(f"  semidiurnal amplitude      {s_b.std():.2f} gpd/in-dia/mile (1 sd)")
print(f"  sub-tidal variability      {seep_rate[m].std():.2f} gpd/in-dia/mile (1 sd)")

#### Figure

In [ ]:
series = (
    (tide, "Coastal water level\nat the MODFLOW boundary, ft",
     "Tidal forcing", "#1f77b4", False),
    (seep_rate, "Net sewer seepage,\ngpd/in-diameter/mile",
     "Groundwater seepage into the sewer (positive = infiltration)",
     "#d62728", True),
    (connected, f"Connected junctions,\n% of {n_connections}",
     "Junctions with the water table above the pipe invert", "#2ca02c", False),
)

with styles.USGSMap():
    fig, axs = plt.subplots(3, 2, figsize=(7.48, 5.67), layout="constrained",
                            sharex="col", gridspec_kw={"width_ratios": [2.0, 1.0]})

    for row, (y, ylab, head, colr, zeroline) in enumerate(series):
        for c, sl in ((0, m), (1, z)):
            ax = axs[row, c]
            ax.plot(t_d[sl], y[sl], lw=0.9, color=colr)
            if zeroline:
                ax.axhline(0.0, lw=0.5, ls="--", color="0.4")
            ax.grid(True, lw=0.3, alpha=0.4)
            ax.set_xlim(t_d[sl][0], t_d[sl][-1])
        axs[row, 0].set_ylabel(ylab)
        styles.heading(axs[row, 0], heading=head)

    styles.heading(axs[0, 1], heading=f"Detail, days {ZOOM_START_D:.0f}"
                                      f"-{ZOOM_START_D + ZOOM_LEN_D:.0f}")
    axs[2, 0].set_xlabel("Time since 2010-01-01 12:00, days")
    axs[2, 1].set_xlabel("Time, days")

    out = fig_ws / f"sewer_tidal_modulation_{resolution}{fig_ext}"
    fig.savefig(out, dpi=300, transparent=transparent)
print("wrote", out)

#### Reading the figure

The semidiurnal modulation is real but second-order: over the window above the
tidal-band seepage amplitude is roughly a quarter of the total variability, so in
the overview it appears as a ripple and only the detail column shows the cycle
plainly. The connected-junction panel carries the clearest semidiurnal signal; its
stepped appearance is correct, being an integer count.

What dominates the overview is the excursion during the surge, when the net
exchange reverses and sewage leaves the pipes into the aquifer before rebounding
above its prior level. For a compound-flooding study that reversal, rather than
the routine tidal cycle, is the consequential behavior.